In [ ]:
!pip -q install telethon pandas pyarrow tqdm

In [ ]:
import os, re, time, asyncio, json, uuid
from math import ceil
from datetime import timezone
import pandas as pd
from tqdm.auto import tqdm
from datetime import datetime, timezone

from pathlib import Path


In [ ]:
from telethon.sync import TelegramClient
from telethon.errors import FloodWaitError, RPCError
from telethon.tl.functions.channels import GetFullChannelRequest
from telethon.tl.types import MessageMediaPhoto, DocumentAttributeVideo, MessageMediaPoll

In [ ]:
api_id = TELEGRAM_API_ID 
api_hash = TELEGRAM_API_HASH
client = TelegramClient('session_name', api_id, api_hash)


In [ ]:
CHANNELS = [
    "moscowtoplive","ru2ch","mash","bazabazon","litvintm","plsbetenderly","instasamkacore","topor","Lepragram",
    "sms_future","whackdoor","bugfeature","aiaiai","bezposhady","Crypto_Woolf","br_dev","kristina_egiazarova14",
    "veronikastepanova20011","klientvsprav","FamilyBots","trendi","etoznakmag","Coin_Post","bankoffo","commerc","pekagame",
]


In [ ]:
TARGET_TOTAL = 11000
CHUNK_SIZE   = 500                 # как часто сбрасывать на диск
OUT_CSV      = "/content/tg_posts.csv"
OUT_PQ_DIR   = "/content/tg_posts_parquet"   # пишем part-файлами
SEEN_KEYS    = "/content/tg_posts_seen.csv"  # хранилище ключей для возобновления

In [ ]:
os.makedirs(OUT_PQ_DIR, exist_ok=True)

# --- загрузка уже сохранённых ключей (для резюма) ---
if os.path.exists(SEEN_KEYS):
    seen_df = pd.read_csv(SEEN_KEYS, dtype=str)
    seen = set(map(tuple, seen_df[["platform","channel_id","message_id"]].values.tolist()))
else:
    seen = set()

# --- служебные ---
url_regex = re.compile(r"https?://\S+")

def iso_utc(dt):
    if dt is None: return None
    if dt.tzinfo is None: dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc).isoformat()

def has_video(msg):
    if not msg or not msg.media: return False
    if getattr(msg, "video", None): return True
    doc = getattr(msg.media, "document", None)
    if doc and getattr(doc, "attributes", None):
        return any(isinstance(a, DocumentAttributeVideo) for a in doc.attributes)
    return False

def has_photo(msg):
    return isinstance(getattr(msg, "media", None), MessageMediaPhoto)

def has_link(msg):
    text = msg.message or ""
    if url_regex.search(text): return True
    return bool(getattr(msg, "entities", None))

def has_poll(msg):
    return isinstance(getattr(msg, "media", None), MessageMediaPoll)

def reactions_sum(msg):
    r = getattr(msg, "reactions", None)
    if not r: return None
    if getattr(r, "results", None):
        return sum(getattr(t, "count", 0) for t in r.results)
    if getattr(r, "totals", None):
        return sum(getattr(t, "count", 0) for t in r.totals)
    return None

def forward_source(msg):
    fwd = getattr(msg, "fwd_from", None)
    if not fwd: return None, None, None
    src_id = getattr(getattr(fwd, "from_id", None), "channel_id", None) or getattr(getattr(fwd, "from_id", None), "user_id", None)
    src_name = getattr(fwd, "from_name", None)
    src_msg_id = getattr(fwd, "channel_post", None)
    return src_id, src_name, src_msg_id

async def participants_count(client, entity):
    try:
        full = await client(GetFullChannelRequest(entity))
        return getattr(full.full_chat, "participants_count", None)
    except RPCError:
        return None

async def safe_iter_messages(client, entity, limit):
    fetched = 0
    while fetched < limit:
        remain = limit - fetched
        try:
            batch = []
            async for m in client.iter_messages(entity, limit=remain):
                batch.append(m)
                if len(batch) >= remain: break
            if not batch: break
            for m in batch: yield m
            fetched += len(batch)
        except FloodWaitError as e:
            await asyncio.sleep(e.seconds + 1)
        except RPCError:
            break

# --- инкрементальное сохранение ---
def coerce_types(df):
    num_cols = ["likes","comments","reposts","followers"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def save_chunk(chunk_records):
    if not chunk_records: return 0
    df = pd.DataFrame.from_records(chunk_records)
    if df.empty: return 0

    # фильтрация уже виденных ключей (межзапусковая дедупликация)
    df_keys = list(map(tuple, df[["platform","channel_id","message_id"]].astype(str).values.tolist()))
    mask_new = [k not in seen for k in df_keys]
    if not any(mask_new): return 0
    df = df[mask_new]
    if df.empty: return 0

    # внутричанковая дедупликация
    df.drop_duplicates(subset=["platform","channel_id","message_id"], inplace=True)

    df = coerce_types(df)

    # CSV append
    write_header = not os.path.exists(OUT_CSV)
    df.to_csv(OUT_CSV, mode="a", index=False, encoding="utf-8-sig", header=write_header)

    # Parquet part-файл
    part_name = f"part-{uuid.uuid4().hex}.parquet"
    df.to_parquet(os.path.join(OUT_PQ_DIR, part_name), index=False)

    # обновляем seen и файл ключей
    new_keys_df = df[["platform","channel_id","message_id"]].astype(str)
    new_keys_df.to_csv(SEEN_KEYS, mode="a", index=False, header=not os.path.exists(SEEN_KEYS))
    for k in map(tuple, new_keys_df.values.tolist()):
        seen.add(k)

    return len(df)


In [ ]:
# --- основной сбор с периодическим сбросом ---
async def main():
    buffer = []
    saved_total = 0
    per_channel = ceil(TARGET_TOTAL / max(1, len(CHANNELS)))

    async with TelegramClient(SESSION, API_ID, API_HASH) as client:
        for ch in tqdm(CHANNELS, desc="Channels"):
            try:
                entity = await client.get_entity(ch)
            except RPCError:
                continue

            subs = await participants_count(client, entity)
            username = getattr(entity, "username", None)
            chan_id = getattr(entity, "id", None)
            chan_title = getattr(entity, "title", None)

            cnt = 0
            async for msg in safe_iter_messages(client, entity, limit=per_channel):
                if msg is None or msg.id is None:
                    continue

                fwd_id, fwd_name, fwd_msg_id = forward_source(msg)
                likes    = reactions_sum(msg)
                comments = getattr(getattr(msg, "replies", None), "replies", None)
                reposts  = getattr(msg, "forwards", None)

                rec = {
                    "platform": "tg",
                    "source_name": chan_title,
                    "source_link": f"https://t.me/{username}" if username else None,
                    "followers": subs,
                    "channel_id": chan_id,
                    "channel_username": username,
                    "message_id": msg.id,
                    "date": iso_utc(msg.date),
                    "initial_date": iso_utc(msg.date),
                    "text": msg.message or "",
                    "text_len": len(msg.message or ""),
                    "likes": likes,
                    "comments": comments,
                    "reposts": reposts,
                    "has_photo": bool(has_photo(msg)),
                    "has_video": bool(has_video(msg)),
                    "has_link": bool(has_link(msg)),
                    "has_poll": bool(has_poll(msg)),
                    "is_ad": None,
                    "image_url": None,
                    "message_link": f"https://t.me/{username}/{msg.id}" if username else None,
                    "is_forwarded": bool(getattr(msg, "fwd_from", None) is not None),
                    "fwd_src_id": fwd_id,
                    "fwd_src_name": fwd_name,
                    "fwd_src_msg_id": fwd_msg_id,
                }
                buffer.append(rec)
                cnt += 1

                if len(buffer) >= CHUNK_SIZE:
                    saved = save_chunk(buffer)
                    buffer.clear()
                    saved_total += saved

                if cnt % 500 == 0:
                    await asyncio.sleep(1)

    # финальный сброс
    saved = save_chunk(buffer)
    saved_total += saved
    print(f"Saved incrementally: {saved_total} rows")
    print(f"CSV: {OUT_CSV}")
    print(f"Parquet dir: {OUT_PQ_DIR}")
    # при необходимости можно собрать head:
    if os.path.exists(OUT_CSV):
        df_head = pd.read_csv(OUT_CSV, nrows=3)
        display(df_head)

await main()

In [ ]:
INPUT_CSV   = "/content/tg_posts.csv"
DATE_TAG    = datetime.utcnow().date().isoformat()   #
OUTPUT_CSV  = "/content/tg_posts2.csv"                # обновляем поверх
OUTPUT_PQ   = "/content/tg_posts2.parquet"
BATCH_IDS   = 200

# === Helpers ===
TME_RE = re.compile(r"(?:https?://)?t\.me/(?:c/)?([A-Za-z0-9_]+)/?")

def extract_username(row):
    # 1) явный username
    u = str(row.get("channel_username", "")).strip()
    if u and u != "None" and u.lower() != "nan":
        return u.lstrip("@")
    # 2) из source_link
    s = str(row.get("source_link", "")).strip()
    m = TME_RE.search(s)
    if m:
        return m.group(1)
    # 3) из message_link
    ml = str(row.get("message_link", "")).strip()
    m2 = TME_RE.search(ml)
    if m2:
        return m2.group(1)
    return None

def reactions_sum(msg):
    r = getattr(msg, "reactions", None)
    if not r: return None
    if getattr(r, "results", None):
        return sum(getattr(t, "count", 0) for t in r.results)
    if getattr(r, "totals", None):
        return sum(getattr(t, "count", 0) for t in r.totals)
    return None

async def fetch_and_update():
    base = pd.read_csv(INPUT_CSV)
    if "message_id" not in base.columns:
        raise ValueError("В INPUT_CSV нет столбца 'message_id'")

    # канонический username
    base["canon_username"] = base.apply(extract_username, axis=1)
    base = base[base["canon_username"].notna()].copy()

    # новые колонки на дату
    col_l = f"likes_{DATE_TAG}"
    col_c = f"comments_{DATE_TAG}"
    col_r = f"reposts_{DATE_TAG}"
    col_t = f"collected_at_{DATE_TAG}"
    for col in (col_l, col_c, col_r, col_t):
        if col not in base.columns:
            base[col] = pd.Series([pd.NA]*len(base))

    # группировка: username -> список message_id
    groups = (
        base[["canon_username","message_id"]]
        .dropna()
        .astype({"message_id":"int64"})
        .groupby("canon_username")["message_id"]
        .apply(list)
        .to_dict()
    )

    async with TelegramClient(SESSION, API_ID, API_HASH) as client:
        for uname, mids in tqdm(groups.items(), desc="Channels"):
            # стараемся резолвить по ссылке (надёжнее)
            try:
                entity = await client.get_entity(f"https://t.me/{uname}")
            except (RPCError, ValueError):
                continue

            # батчами по ids
            for i in range(0, len(mids), BATCH_IDS):
                batch = mids[i:i+BATCH_IDS]
                # повтор при FloodWait
                while True:
                    try:
                        msgs = await client.get_messages(entity, ids=batch)
                        break
                    except FloodWaitError as e:
                        await asyncio.sleep(e.seconds + 1)
                    except RPCError:
                        msgs = []
                        break

                now_iso = datetime.now(timezone.utc).isoformat()

                for msg in msgs:
                    if msg is None:
                        continue
                    # точная маска: username + message_id
                    mask = (base["canon_username"] == uname) & (base["message_id"].astype(int) == int(msg.id))

                    # новые значения
                    val_l = reactions_sum(msg)
                    val_c = getattr(getattr(msg, "replies", None), "replies", None)
                    val_r = getattr(msg, "forwards", None)

                    # дозаполнение только NaN
                    base.loc[mask & base[col_l].isna(), col_l] = val_l
                    base.loc[mask & base[col_c].isna(), col_c] = val_c
                    base.loc[mask & base[col_r].isna(), col_r] = val_r
                    base.loc[mask & base[col_t].isna(), col_t] = now_iso

    # типы
    for c in [col_l, col_c, col_r, "likes", "comments", "reposts", "followers"]:
        if c in base.columns:
            base[c] = pd.to_numeric(base[c], errors="coerce")

    # запись
    base.drop(columns=["canon_username"], inplace=True)
    base.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    base.to_parquet(OUTPUT_PQ, index=False)
    print(f"Updated: {OUTPUT_CSV}")
    print(f"Parquet: {OUTPUT_PQ}")
    print("New columns:", col_l, col_c, col_r, col_t)

await fetch_and_update()

In [ ]:

# --- пути ---
POSTS_CSV = "/content/tg_posts.csv"      # твой общий постовый датасет TG
PAIRS_CSV = "/content/tg_pairs.csv"      # файл с original_* и forward_*
OUT_CSV   = "/content/tg_merged.csv"
OUT_PQ    = "/content/tg_merged.parquet"

# --- утилиты ---
TME_RE = re.compile(r"(?:https?://)?t\.me/(?:c/)?([A-Za-z0-9_]+)/(\d+)")

def parse_tme(link: str):
    if not isinstance(link, str):
        return None, None
    m = TME_RE.search(link.strip())
    if not m:
        return None, None
    return m.group(1), int(m.group(2))

def norm_username(u):
    if u is None:
        return None
    s = str(u).strip()
    if not s or s.lower() in ("none", "nan"):
        return None
    return s.lstrip("@")

def safe_int(x):
    try:
        return int(float(x))
    except Exception:
        return None

# --- загрузка ---
posts = pd.read_csv(POSTS_CSV)
pairs = pd.read_csv(PAIRS_CSV)

# --- ключи в posts: username + message_id ---
# 1) username из channel_username или из source_link/message_link
def extract_username_posts(row):
    u = norm_username(row.get("channel_username"))
    if u:
        return u
    for col in ("source_link", "message_link"):
        u2, _ = parse_tme(row.get(col, ""))
        if u2:
            return u2
    return None

posts["__uname__"] = posts.apply(extract_username_posts, axis=1)
posts["__mid__"]   = posts["message_id"].apply(safe_int) if "message_id" in posts.columns else None

# отфильтруем, где ключ собрать удалось
posts_keyed = posts[posts["__uname__"].notna() & posts["__mid__"].notna()].copy()

# --- ключи в pairs: берём из original_post_link, иначе из original_channel + original_post_id ---
def extract_uname_mid_pairs(row):
    u, m = parse_tme(row.get("original_post_link", ""))
    if u and m:
        return u, m
    u2 = norm_username(row.get("original_channel"))
    m2 = safe_int(row.get("original_post_id"))
    return (u2, m2)

pairs["__uname__"], pairs["__mid__"] = zip(*pairs.apply(extract_uname_mid_pairs, axis=1))
pairs_keyed = pairs[pairs["__uname__"].notna() & pairs["__mid__"].notna()].copy()

# --- выделим столбцы из pairs, которые нужно добавить в конец ---
# берём всё, что относится к форвардам и метрикам сопоставления
forward_cols = [c for c in pairs_keyed.columns if c.startswith("forward_")]
meta_cols = [c for c in [
    "reason","similarity","viewsCount","sharesCount","commentsCount",
    "reactionsCount","forwardsCount","mentionsCount"
] if c in pairs_keyed.columns]

# также захватим любые уже рассчитанные original_* снапшоты (на дату), если есть
original_snapshot_cols = [c for c in pairs_keyed.columns if c.startswith("original_")]

cols_to_add = original_snapshot_cols + forward_cols + meta_cols

pairs_slim = pairs_keyed[["__uname__","__mid__"] + cols_to_add].copy()

# если в pairs дубли по ключу — оставим первый (или агрегируй по необходимости)
pairs_slim = pairs_slim.drop_duplicates(subset=["__uname__","__mid__"], keep="first")

# --- merge (left) ---
merged = posts_keyed.merge(pairs_slim, on=["__uname__","__mid__"], how="left", suffixes=("",""))

# признак наличия пары
merged["has_forward_pair"] = merged[forward_cols].notna().any(axis=1) if forward_cols else False

# --- порядок столбцов: исходные posts -> затем добавляемые из pairs ---
base_cols = [c for c in posts.columns]  # в исходном порядке
add_cols  = [c for c in original_snapshot_cols + forward_cols + meta_cols if c in merged.columns]
final_cols = base_cols + add_cols + (["has_forward_pair"] if "has_forward_pair" in merged.columns else [])

# могут быть технические ключи в base_cols; уберём их из финального списка и добавим в конец для отладки
tech_cols = ["__uname__","__mid__"]
final_cols = [c for c in final_cols if c not in tech_cols] + [c for c in tech_cols if c in merged.columns]

merged = merged[final_cols]

# --- сохранение ---
merged.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
merged.to_parquet(OUT_PQ, index=False)

print(f"Saved: {OUT_CSV}")
print(f"Saved: {OUT_PQ}")
print("Rows:", len(merged))

In [ ]:
# WIDE из TG+VK с сохранением исходных полей и ER по репостам.
import os, re, numpy as np, pandas as pd

# --- пути (проверь имена файлов) ---
TG_CSV = "/content/tg_merged.csv"
VK_CSV = "/content/vk_posts.csv"   # твой VK CSV
OUT_CSV = "/content/social_wide.csv"
OUT_PQ  = "/content/social_wide.parquet"

# --- регэксп для срезов ---
SNAP_PAT = re.compile(r"^(?:original_)?(likes|comments|reposts)_(\d{4}-\d{2}-\d{2})$")
COLLECTED_PAT = re.compile(r"^(?:original_)?collected_at_(\d{4}-\d{2}-\d{2})$")

def read_df(path):
    if not os.path.exists(path): return pd.DataFrame()
    return pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)

# исходные поля, которые хотим сохранить как есть
VK_KEEP = [
    "group_id","post_id","from_id","text","date","initial_date","likes",
    "comments","reposts","image_url","followers","source_name","source_link",
    "is_ad","has_photo","has_video","has_link","has_poll"
]
TG_KEEP = [
    "platform","source_name","source_link","followers","channel_id","channel_username",
    "message_id","date","initial_date","text","text_len","likes","comments","reposts",
    "has_photo","has_video","has_link","has_poll","is_ad","image_url","message_link",
    "is_forwarded","fwd_src_id","fwd_src_name","fwd_src_msg_id"
]

def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = pd.NA
    return df

def base_block_vk(vk):
    vk = ensure_cols(vk, VK_KEEP)
    out = pd.DataFrame({
        # базовые
        "platform":     "vk",
        "source_name":  vk["source_name"],
        "source_link":  vk["source_link"],
        "followers":    pd.to_numeric(vk["followers"], errors="coerce"),
        "post_id":      vk["post_id"],
        "published_at": vk["initial_date"].fillna(vk["date"]),
        "text":         vk["text"],
        "has_photo":    vk["has_photo"],
        "has_video":    vk["has_video"],
        "has_link":     vk["has_link"],
        "has_poll":     vk["has_poll"],
        "is_ad":        vk["is_ad"],
        "link":         vk["source_link"],
        # оставляем оригинальные поля VK
        "vk_group_id":  vk["group_id"],
        "vk_from_id":   vk["from_id"],
        "image_url":    vk["image_url"],
    })
    return out

def base_block_tg(tg):
    tg = ensure_cols(tg, TG_KEEP)
    out = pd.DataFrame({
        "platform":     "tg",
        "source_name":  tg["source_name"],
        "source_link":  tg["source_link"],
        "followers":    pd.to_numeric(tg["followers"], errors="coerce"),
        "post_id":      tg["message_id"],
        "published_at": tg["initial_date"].fillna(tg["date"]),
        "text":         tg["text"],
        "has_photo":    tg["has_photo"],
        "has_video":    tg["has_video"],
        "has_link":     tg["has_link"],
        "has_poll":     tg["has_poll"],
        "is_ad":        tg["is_ad"],
        "link":         tg["message_link"].fillna(tg["source_link"]),
        # оставляем важные TG-поля
        "tg_channel_id":       tg["channel_id"],
        "tg_channel_username": tg["channel_username"],
        "tg_is_forwarded":     tg["is_forwarded"],
        "tg_fwd_src_id":       tg["fwd_src_id"],
        "tg_fwd_src_name":     tg["fwd_src_name"],
        "tg_fwd_src_msg_id":   tg["fwd_src_msg_id"],
        "image_url":           tg["image_url"],
    })
    return out

def collect_snapshot_cols(df):
    snaps = {}         # {date: {"likes": col, "comments": col, "reposts": col}}
    collected = {}     # {date: col}
    for c in df.columns:
        m = SNAP_PAT.match(str(c))
        if m:
            metric, d = m.group(1), m.group(2)
            snaps.setdefault(d, {})[metric] = c
            continue
        k = COLLECTED_PAT.match(str(c))
        if k:
            d = k.group(1)
            collected[d] = c
    return snaps, collected

def attach_snapshots(base, raw, snaps, collected):
    out = base.copy()
    for d, mapping in sorted(snaps.items()):
        for metric, col in mapping.items():
            out[f"{metric}_{d}"] = pd.to_numeric(raw[col], errors="coerce")
        # если есть collected_at_* — добавим текстовым столбцом
        ccol = collected.get(d)
        if ccol:
            out[f"collected_at_{d}"] = raw[ccol].astype(str)
    return out

def compute_er(wide):
    # er_<date> = reposts_<date> / followers
    dates = []
    for c in wide.columns:
        m = re.match(r"^reposts_(\d{4}-\d{2}-\d{2})$", c)
        if m: dates.append(m.group(1))
    dates = sorted(set(dates))
    for d in dates:
        num = pd.to_numeric(wide.get(f"reposts_{d}"), errors="coerce")
        den = pd.to_numeric(wide.get("followers"), errors="coerce")
        wide[f"er_{d}"] = np.where((den > 0) & num.notna(), num/den, np.nan)
    # итоговый engagement_rate = по самой поздней дате с ненулевым значением
    if dates:
        # идём по датам в хронологическом порядке, перезаписывая на более поздние
        latest = pd.Series(np.nan, index=wide.index, dtype=float)
        for d in dates:
            v = wide[f"er_{d}"]
            latest = np.where(~np.isnan(v), v, latest)
        wide["engagement_rate"] = latest
    return dates, wide

# --- загрузка ---
tg_raw = read_df(TG_CSV)
vk_raw = read_df(VK_CSV)

frames = []

# TG
if not tg_raw.empty:
    tg_base = base_block_tg(tg_raw)
    tg_snaps, tg_collected = collect_snapshot_cols(tg_raw)
    tg_wide = attach_snapshots(tg_base, tg_raw, tg_snaps, tg_collected)
    frames.append(tg_wide)

# VK
if not vk_raw.empty:
    vk_base = base_block_vk(vk_raw)
    vk_snaps, vk_collected = collect_snapshot_cols(vk_raw)
    vk_wide = attach_snapshots(vk_base, vk_raw, vk_snaps, vk_collected)
    frames.append(vk_wide)

# объединение
wide = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# ER
snapshot_dates, wide = compute_er(wide)

# типобезопасность перед parquet
# строки — явно как string; булевые — в bool; числа — numeric
for col in ["platform","source_name","source_link","published_at","text","link",
            "tg_channel_username","tg_fwd_src_name","image_url"]:
    if col in wide.columns:
        wide[col] = wide[col].astype("string")

for col in ["has_photo","has_video","has_link","has_poll","is_ad","tg_is_forwarded"]:
    if col in wide.columns:
        # аккуратно приводим к bool через map
        wide[col] = wide[col].map(lambda x: bool(x) if pd.notna(x) else pd.NA)

num_like = [c for c in wide.columns if re.match(r"^(likes|comments|reposts|er)_\d{4}-\d{2}-\d{2}$", c)]
for col in ["followers"] + num_like + ["engagement_rate"]:
    if col in wide.columns:
        wide[col] = pd.to_numeric(wide[col], errors="coerce")

# сохранение
wide.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
wide.to_parquet(OUT_PQ, index=False)

# отчёт
dates_found = sorted({m.group(2) for col in wide.columns for m in [SNAP_PAT.match(col)] if m})
print("Saved:", OUT_CSV)
print("Saved:", OUT_PQ)
print("Shape:", wide.shape)
print("Snapshot dates:", dates_found)
print(wide.head(3))
print(wide.info())

